<a href="https://colab.research.google.com/github/MohammedAyman07/curriculum-rag-research/blob/main/notebooks/04_hybrid_retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CurriculumRAG — Hybrid Retrieval

## Objective
Combine keyword-based and dense semantic retrieval.

## Fusion Method
Reciprocal Rank Fusion (RRF)

## Retrieval Methods
1. TF-IDF
2. Dense Semantic Retrieval
3. Hybrid RRF

## Evaluation
- Precision@5
- Recall@5
- Mean Reciprocal Rank
- Median Query Latency

In [1]:

%pip -q install -U sentence-transformers

import numpy as np
import pandas as pd
import torch
import time

from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

BASE_URL = (
    "https://raw.githubusercontent.com/"
    "MohammedAyman07/curriculum-rag-research/main/data/"
)

passages_df = pd.read_csv(BASE_URL + "passages.csv")
queries_df = pd.read_csv(BASE_URL + "queries.csv")

assert len(passages_df) == 40
assert len(queries_df) == 60

documents = passages_df["text"].fillna("").astype(str).tolist()
document_ids = passages_df["doc_id"].astype(str).tolist()

dev_queries = queries_df[
    queries_df["split"] == "dev"
].copy()

assert len(dev_queries) == 15

print("Dataset loaded!")
print("Passages:", len(documents))
print("Development questions:", len(dev_queries))
print("Test questions remain held out.")


Dataset loaded!
Passages: 40
Development questions: 15
Test questions remain held out.


In [2]:

# Build TF-IDF index

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True,
    token_pattern=r"(?u)\b\w+\b"
)

tfidf_start = time.perf_counter()

tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

tfidf_index_time = time.perf_counter() - tfidf_start

# Load the same embedding model as the dense experiment
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Use CPU to match the previous dense experiment's device
DEVICE = "cpu"

model_start = time.perf_counter()

model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE
)

model_load_time = time.perf_counter() - model_start

# Generate normalized document embeddings
embedding_start = time.perf_counter()

if hasattr(model, "encode_document"):
    document_embeddings = model.encode_document(
        documents,
        batch_size=32,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    )
else:
    document_embeddings = model.encode(
        documents,
        batch_size=32,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    )

document_embeddings = np.asarray(
    document_embeddings,
    dtype=np.float32
)

document_embedding_time = (
    time.perf_counter() - embedding_start
)

assert document_embeddings.shape[0] == len(documents)

print("Both indexes are ready!")
print("TF-IDF matrix:", tfidf_matrix.shape)
print("Dense matrix:", document_embeddings.shape)
print("TF-IDF index time:", round(tfidf_index_time, 4), "seconds")
print("Model loading time:", round(model_load_time, 3), "seconds")
print("Document embedding time:", round(document_embedding_time, 3), "seconds")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Both indexes are ready!
TF-IDF matrix: (40, 2195)
Dense matrix: (40, 384)
TF-IDF index time: 0.1365 seconds
Model loading time: 5.133 seconds
Document embedding time: 3.381 seconds


In [3]:

# Hybrid Retrieval using Reciprocal Rank Fusion

RRF_CONSTANT = 60

def get_tfidf_ranking(question):

    query_vector = tfidf_vectorizer.transform([question])

    scores = (
        query_vector @ tfidf_matrix.T
    ).toarray().flatten()

    indices = np.argsort(
        -scores,
        kind="stable"
    )

    return [document_ids[i] for i in indices]


def get_dense_ranking(question):

    if hasattr(model, "encode_query"):
        query_embedding = model.encode_query(
            [question],
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False
        )[0]
    else:
        query_embedding = model.encode(
            [question],
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False
        )[0]

    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32
    )

    scores = document_embeddings @ query_embedding

    indices = np.argsort(
        -scores,
        kind="stable"
    )

    return [document_ids[i] for i in indices]


def reciprocal_rank_fusion(
    rankings,
    rrf_constant=RRF_CONSTANT
):

    fused_scores = {
        doc_id: 0.0 for doc_id in document_ids
    }

    for ranking in rankings:

        for rank, doc_id in enumerate(ranking, start=1):

            fused_scores[doc_id] += (
                1.0 / (rrf_constant + rank)
            )

    # Stable tie-breaking by original document order
    doc_order = {
        doc_id: i for i, doc_id in enumerate(document_ids)
    }

    final_ranking = sorted(
        document_ids,
        key=lambda doc_id: (
            -fused_scores[doc_id],
            doc_order[doc_id]
        )
    )

    return final_ranking, fused_scores


def retrieve_hybrid(question, top_k=5):

    start = time.perf_counter()

    tfidf_ranking = get_tfidf_ranking(question)
    dense_ranking = get_dense_ranking(question)

    fused_ranking, fused_scores = reciprocal_rank_fusion(
        [tfidf_ranking, dense_ranking]
    )

    results = []

    for doc_id in fused_ranking[:top_k]:

        index = document_ids.index(doc_id)

        results.append({
            "doc_id": doc_id,
            "rrf_score": fused_scores[doc_id],
            "topic": passages_df.iloc[index]["topic"],
            "text": documents[index]
        })

    latency_ms = (
        time.perf_counter() - start
    ) * 1000

    return results, latency_ms


# Test the hybrid system

sample_question = "How does binary search reduce the search space?"

sample_results, sample_latency = retrieve_hybrid(
    sample_question
)

print("Question:", sample_question)
print(f"Hybrid latency: {sample_latency:.3f} ms\n")

for rank, result in enumerate(sample_results, start=1):
    print(f"Rank {rank}: {result['doc_id']}")
    print("Topic:", result["topic"])
    print("RRF score:", round(result["rrf_score"], 6))
    print()


Question: How does binary search reduce the search space?
Hybrid latency: 22.541 ms

Rank 1: P033
Topic: Binary Search
RRF score: 0.032787

Rank 2: P014
Topic: Binary Search Trees
RRF score: 0.032258

Rank 3: P002
Topic: Space Complexity
RRF score: 0.03125

Rank 4: P034
Topic: Hashing
RRF score: 0.03009

Rank 5: P032
Topic: Linear Search
RRF score: 0.029762



In [4]:

# Evaluation metrics

def calculate_metrics(retrieved_ids, relevant_ids, k=5):

    relevant = set(relevant_ids)

    if not relevant:
        raise ValueError("No relevant passage labels found.")

    retrieved = retrieved_ids[:k]
    hits = set(retrieved) & relevant

    precision = len(hits) / k
    recall = len(hits) / len(relevant)

    reciprocal_rank = 0.0

    for rank, doc_id in enumerate(retrieved, start=1):
        if doc_id in relevant:
            reciprocal_rank = 1 / rank
            break

    return precision, recall, reciprocal_rank


# Warm-up
retrieve_hybrid("What is a binary search tree?")

evaluation_records = []

for _, row in dev_queries.iterrows():

    results, _ = retrieve_hybrid(
        row["question"],
        top_k=5
    )

    retrieved_ids = [
        item["doc_id"] for item in results
    ]

    relevant_ids = str(
        row["relevant_doc_ids"]
    ).split("|")

    precision, recall, mrr = calculate_metrics(
        retrieved_ids,
        relevant_ids,
        k=5
    )

    latency_samples = [
        retrieve_hybrid(row["question"], top_k=5)[1]
        for _ in range(5)
    ]

    evaluation_records.append({
        "query_id": row["query_id"],
        "question": row["question"],
        "relevant_doc_ids": row["relevant_doc_ids"],
        "retrieved_doc_ids": "|".join(retrieved_ids),
        "precision_at_5": precision,
        "recall_at_5": recall,
        "mrr": mrr,
        "median_query_latency_ms": float(
            np.median(latency_samples)
        )
    })

hybrid_results_df = pd.DataFrame(evaluation_records)

print("Hybrid development evaluation completed!")
display(hybrid_results_df)


Hybrid development evaluation completed!


,query_id,question,relevant_doc_ids,retrieved_doc_ids,precision_at_5,recall_at_5,mrr,median_query_latency_ms
0,Q001,How does an algorithm's running time change as...,P001,P001|P002|P040|P028|P004,0.2,1.0,1.0,17.671987
1,Q004,Why can an array element be accessed directly ...,P004,P004|P034|P026|P028|P011,0.2,1.0,1.0,29.397195
2,Q006,Which linked list supports traversal in both d...,P006,P006|P005|P007|P016|P036,0.2,1.0,1.0,67.711974
3,Q008,Which principle explains why the latest stack ...,P008,P008|P009|P010|P026|P004,0.2,1.0,1.0,89.615995
4,Q010,Where does a queue insert new elements and rem...,P010,P010|P011|P040|P004|P025,0.2,1.0,1.0,60.158064
5,Q013,How many children can a node have in a binary ...,P013,P013|P014|P015|P016|P033,0.2,1.0,1.0,55.591166
6,Q015,What balance condition does an AVL tree mainta...,P015,P015|P014|P016|P023|P013,0.2,1.0,1.0,67.551138
7,Q018,"What are the main components of a graph, and h...",P017,P017|P021|P018|P019|P020,0.2,1.0,1.0,64.431016
8,Q020,Which graph traversal visits nearby vertices b...,P019,P019|P016|P020|P017|P018,0.2,1.0,1.0,55.870109
9,Q022,How does DFS explore a graph and then return t...,P020,P019|P020|P018|P017|P023,0.2,1.0,0.5,72.525191


In [5]:

# Hybrid experiment summary

hybrid_summary = {
    "Method": "Hybrid RRF",
    "Model": MODEL_NAME,
    "RRF Constant": RRF_CONSTANT,
    "Device": DEVICE,
    "Precision@5": hybrid_results_df["precision_at_5"].mean(),
    "Recall@5": hybrid_results_df["recall_at_5"].mean(),
    "MRR": hybrid_results_df["mrr"].mean(),
    "Median Query Latency (ms)": (
        hybrid_results_df["median_query_latency_ms"].median()
    ),
    "TF-IDF Index Time (seconds)": tfidf_index_time,
    "Model Loading Time (seconds)": model_load_time,
    "Document Embedding Time (seconds)": document_embedding_time
}

hybrid_summary_df = pd.DataFrame([hybrid_summary])

display(hybrid_summary_df)

# Save results
RESULTS_DIR = Path("/content/curriculum-rag-results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

hybrid_results_df.to_csv(
    RESULTS_DIR / "hybrid_dev_predictions.csv",
    index=False
)

hybrid_summary_df.to_csv(
    RESULTS_DIR / "hybrid_dev_metrics.csv",
    index=False
)

print("Hybrid experiment results saved successfully!")


,Method,Model,RRF Constant,Device,Precision@5,Recall@5,MRR,Median Query Latency (ms),TF-IDF Index Time (seconds),Model Loading Time (seconds),Document Embedding Time (seconds)
0,Hybrid RRF,sentence-transformers/all-MiniLM-L6-v2,60,cpu,0.2,1.0,0.966667,55.870109,0.136504,5.133005,3.380559


Hybrid experiment results saved successfully!
